In [0]:
# Cell 1
%pip install prophet

In [0]:
# Cell 2
import os, sys, uuid
sys.path.insert(0, os.path.abspath(".."))

from ecom.audit import Audit
from ecom.forecast import run_forecast_scoring

dbutils.widgets.text("catalog", "ecom_dev")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("bundle_target", "dev")

catalog = dbutils.widgets.get("catalog")
run_id = dbutils.widgets.get("run_id") or str(uuid.uuid4())
bundle_target = dbutils.widgets.get("bundle_target")
triggered_by = spark.sql("SELECT current_user()").first()[0]
print(f"catalog={catalog} run_id={run_id}")

In [0]:
# Cell 3
audit = Audit(spark, catalog, run_id, "forecast_score", bundle_target, triggered_by)
run_forecast_scoring(spark, catalog, run_id, audit)

In [0]:
%sql
SELECT grain, forecast_date, horizon_days,
       round(yhat) AS forecast, round(yhat_lower) AS low, round(yhat_upper) AS high
FROM ecom_dev.ml.sales_forecast
WHERE grain = 'month'
ORDER BY forecast_date;

In [0]:
%sql
SELECT forecast_run_id, model_version, generated_at, count(*) AS rows
FROM ecom_dev.ml.sales_forecast
GROUP BY forecast_run_id, model_version, generated_at
ORDER BY generated_at;